# Lab #28 — Phiên bản chạy TRỰC TIẾP trên Kaggle (All-in-One)
**SV:** Vũ Quang Bảo — **MSV:** 2A202600610

Notebook này chạy **toàn bộ mini AI-platform ngay trong 1 Kaggle notebook** trên GPU T4,
không cần Docker/local. Gồm: vLLM serving + embedding + Qdrant (in-memory) +
RAG API Gateway + MLflow tracking + smoke test. Bật **GPU T4 x2** trước khi chạy.

> Đây là bản 'chạy bằng Kaggle'. Bản hybrid đầy đủ (Kafka/Prefect/Grafana) chạy ở local,
> xem `docker-compose.yml`; notebook `lab28_kaggle_gpu.ipynb` là bản chỉ expose GPU qua ngrok.


## Cell 1 — Cài dependencies


In [ ]:
!pip install -q vllm fastapi uvicorn sentence-transformers qdrant-client mlflow httpx nest_asyncio
# Fallback nếu vLLM lỗi version: !pip install -q transformers==4.46.3 vllm==0.7.3


## Cell 2 — Khởi động vLLM server (GPU) nền


In [ ]:
import subprocess, threading, time, requests

MODEL = 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4'

def run_vllm():
    subprocess.run([
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL, '--port', '8001',
        '--max-model-len', '4096', '--gpu-memory-utilization', '0.55',
    ])

threading.Thread(target=run_vllm, daemon=True).start()

def wait_ready(url, n=90):
    for _ in range(n):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(5)
    return False

print('vLLM ready:', wait_ready('http://localhost:8001/health'))


## Cell 3 — Embedding model + Qdrant in-memory (Vector Store)


In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

emb_model = SentenceTransformer('BAAI/bge-small-en-v1.5')  # 384 dims
VECTOR_SIZE = emb_model.get_sentence_embedding_dimension()

qdrant = QdrantClient(':memory:')  # Qdrant nhúng, không cần server riêng
qdrant.recreate_collection('documents',
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE))
print('Qdrant in-memory sẵn sàng, dim =', VECTOR_SIZE)


## Cell 4 — Ingest + Embed dữ liệu mẫu vào Qdrant (Integration 1→5 rút gọn)


In [ ]:
DOCS = [
    {'id': 'doc_001', 'text': 'AI platform integration test'},
    {'id': 'doc_002', 'text': 'Kafka to Airflow pipeline'},
    {'id': 'doc_003', 'text': 'Event-driven architecture decouples producers and consumers'},
    {'id': 'doc_004', 'text': 'Platform engineering builds paved-road infrastructure for ML teams'},
    {'id': 'doc_005', 'text': 'Observability relies on metrics, logs and traces'},
]
vecs = emb_model.encode([d['text'] for d in DOCS]).tolist()
qdrant.upsert('documents', points=[
    PointStruct(id=i, vector=v, payload=d) for i, (v, d) in enumerate(zip(vecs, DOCS))
])
print('Đã nạp', len(DOCS), 'vectors vào Qdrant')


## Cell 5 — RAG API Gateway (FastAPI) chạy trong notebook


In [ ]:
import httpx, os, time
from fastapi import FastAPI
from pydantic import BaseModel, Field
import uvicorn, nest_asyncio, threading
nest_asyncio.apply()

app = FastAPI(title='Kaggle AI Platform Gateway')

class ChatReq(BaseModel):
    query: str = Field(..., min_length=1)
    top_k: int = 3

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/api/v1/chat')
def chat(r: ChatReq):
    start = time.time()
    qv = emb_model.encode([r.query])[0].tolist()          # 1) embed
    hits = qdrant.search('documents', query_vector=qv, limit=r.top_k)  # 2) vector search
    ctx = [h.payload['text'] for h in hits]
    prompt = f'Context: {ctx}\n\nQuery: {r.query}'
    try:                                                   # 3) LLM + fallback
        resp = httpx.post('http://localhost:8001/v1/chat/completions', json={
            'model': MODEL, 'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': 256}, timeout=60)
        ans = resp.json()['choices'][0]['message']['content']; degraded = False
    except Exception as e:
        ans = f'[fallback] LLM unreachable: {e}'; degraded = True
    return {'answer': ans, 'latency_ms': round((time.time()-start)*1000, 2),
            'degraded': degraded, 'context_hits': len(ctx)}

threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000),
                 daemon=True).start()
time.sleep(5); print('Gateway chạy tại http://localhost:8000')


## Cell 6 — MLflow tracking (Integration 6 & 7)


In [ ]:
import mlflow
mlflow.set_tracking_uri('file:///kaggle/working/mlruns')
mlflow.set_experiment('lab28-integration')
with mlflow.start_run(run_name='kaggle-allinone-v1'):
    mlflow.log_param('model', MODEL)
    mlflow.log_param('vector_size', VECTOR_SIZE)
    mlflow.log_metric('gpu_memory_utilization', 0.55)
    mlflow.set_tag('status', 'production')
print('MLflow logged -> /kaggle/working/mlruns')


## Cell 7 — Smoke test end-to-end (RAG thật trên GPU)


In [ ]:
import httpx
r = httpx.post('http://localhost:8000/api/v1/chat',
    json={'query': 'Explain event-driven architecture for AI platforms'}, timeout=90)
d = r.json()
print('status     :', r.status_code)
print('latency_ms :', d['latency_ms'])
print('context_hits:', d['context_hits'], '| degraded:', d['degraded'])
print('answer     :', d['answer'][:400])
assert r.status_code == 200 and len(d['answer']) > 10
print('\nSMOKE TEST PASSED — RAG end-to-end chạy trên Kaggle GPU')


## Cell 8 — (Tùy chọn) Expose gateway ra ngoài qua ngrok
Để máy local / người chấm gọi vào API Gateway đang chạy trên Kaggle.


In [ ]:
# !pip install -q pyngrok
# from pyngrok import ngrok
# import os, getpass
# def get_ngrok_token():
#     try:
#         from kaggle_secrets import UserSecretsClient
#         token = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
#         if token:
#             return token
#     except Exception:
#         pass
#     return os.environ.get('NGROK_AUTHTOKEN') or getpass.getpass('NGROK_AUTHTOKEN: ')
# ngrok.set_auth_token(get_ngrok_token())
# print('PUBLIC URL:', ngrok.connect(8000, 'http').public_url)
